# Sudoku Solver

Alex Keyken 13/11/26

Simple sudoku solver that fills cells where only one digit is possible and recursively guesses otherwise. Code was written without use of AI.

To Do:
- Add image recognition.
- Add CNN for number detection.

In [106]:
N_BOX_ROWS = 3
N_BOX_COLS = N_BOX_ROWS
N_BOARD_ROWS = N_BOX_ROWS*N_BOX_COLS
N_BOARD_COLS = N_BOARD_ROWS

BOXES_PER_ROW = N_BOARD_ROWS//N_BOX_ROWS
BOXES_PER_COL = N_BOARD_COLS//N_BOX_COLS

def print_board(board: list[list[int]]) -> None:
    """Visualise board with box divider lines."""
    for iline in range(len(board)):
        if iline % N_BOX_ROWS == 0 and iline != 0:
            print("- " * (len(board) + BOXES_PER_COL - 1))
        for ichar in range(len(board[0])):
            if ichar % N_BOX_COLS == 0 and ichar != 0:
                print("|", end=" ")
            print(board[iline][ichar], end=" ")
        print()

def get_box_digits(board: list[list[int]], cell: tuple[int, int]) -> list[int]:
    """Retrieves a list containing all digits inside the box that the cell belongs to."""
    box_digit_list = []
    row, col = cell
    box_row_offset, box_col_offset = row//N_BOX_ROWS, col//N_BOX_COLS
    for r in range(N_BOX_ROWS):
        for k in range(N_BOX_COLS):
            box_digit_list.append(board[N_BOX_ROWS*box_row_offset+r][N_BOX_COLS*box_col_offset+k])
    
    return box_digit_list

def compute_cell_candidates(board: list[list[int]], cell: tuple[int, int]) -> list[int]:
    """Computes candidates (digits that would not cause rule breaks) for a cell."""
    cell_candidate_list = set([x for x in range(1, N_BOARD_ROWS+1)])
    row, col = cell
    
    cell_candidate_list -= set(board[row])
    cell_candidate_list -= set([board[row_index][col] for row_index in range(len(board))])
    cell_candidate_list -= set(get_box_digits(board, cell))
    
    return list(cell_candidate_list)

def compute_board_candidates(board: list[list[int]]) -> dict[tuple[int, int], list[int]]:
    """
    Creates a dictionary containing a list of candidates per cell on the board.
    Indexed by the cell coordinates such that: dict[(row,col)] = [list of digits].
    """
    candidate_dict = {}
    for row in range(len(board)):
        for col in range(len(board[row])):
            if board[row][col] == 0:
                cell = (row, col)
                candidate_dict[cell] = compute_cell_candidates(board, cell)
                
    return candidate_dict

def fill_singletons(board: list[list[int]]) -> None:
    """
    Updates board in place by filling singletons (empty cells with only one candidate digit).
    Loops until no more singletons are found.
    """
    made_changes = True
    while made_changes:
        made_changes = False
        candidate_dict = compute_board_candidates(board)
        for cell, candidates in candidate_dict.items():
            if len(candidates) == 1:
                if candidates[0] not in compute_cell_candidates(board, cell):
                    continue
                row, col = cell
                board[row][col] = candidates[0]
                made_changes = True
                
def solve(board: list[list[int]]) -> bool:
    """
    Solves board in place by filling singletons and guessing otherwise,
    recursively until board is full.
    """
    fill_singletons(board)
    
    candidates = compute_board_candidates(board)
    if not candidates:
        return check_solved_board(board)

    most_constrained_cell = min(candidates, key=lambda x: len(candidates[x]))
    
    row, col = most_constrained_cell
    for digit in candidates[most_constrained_cell]:
        trial = [r[:] for r in board]
        trial[row][col] = digit
        
        if solve(trial):
            board[:] = trial
            return True
            
    return False

def check_solved_board(board: list[list[int]]) -> bool:
    """Check completed board for repeated digits within each row, column and box."""
    
    for row in board:
        if len(row) != len(set(row)):
            return False

    transposed_board = list(map(list, zip(*board)))
    for col in transposed_board:
        if len(col) != len(set(col)):
            return False

    for box_row in range(BOXES_PER_ROW):
        for box_col in range(BOXES_PER_COL):
            box = get_box_digits(board, (N_BOX_ROWS*box_row, N_BOX_COLS*box_col))
            if len(box) != len(set(box)):
                return False     
                
    return True

In [107]:
import time

N_BOX_ROWS = 3
N_BOX_COLS = N_BOX_ROWS
N_BOARD_ROWS = N_BOX_ROWS*N_BOX_COLS
N_BOARD_COLS = N_BOARD_ROWS

BOXES_PER_ROW = N_BOARD_ROWS//N_BOX_ROWS
BOXES_PER_COL = N_BOARD_COLS//N_BOX_COLS

def run_test(test_name,board: list[list[int]]) -> None:
    """Solve a board, then print the time taken, validity and result."""
    start = time.perf_counter()
    solved = solve(board)
    elapsed = time.perf_counter() - start
    print(f"{test_name}: solved = {solved}, valid = {check_solved_board(board)}, {elapsed:.3f}s \n")
    if solved:
        print_board(board)
    print()

invalid_sudoku = [
    [5,5,0,4,6,7,3,0,9],
    [9,0,3,8,1,0,4,2,7],
    [1,7,4,2,0,3,0,0,0],
    [2,3,1,9,7,6,8,5,4],
    [8,5,7,1,2,4,0,9,0],
    [4,9,6,3,0,8,1,7,2],
    [0,0,0,0,8,9,2,6,0],
    [7,8,2,6,4,1,0,0,5],
    [0,1,0,0,0,0,7,0,8]
]

easy_sudoku = [
    [5,0,0,4,6,7,3,0,9],
    [9,0,3,8,1,0,4,2,7],
    [1,7,4,2,0,3,0,0,0],
    [2,3,1,9,7,6,8,5,4],
    [8,5,7,1,2,4,0,9,0],
    [4,9,6,3,0,8,1,7,2],
    [0,0,0,0,8,9,2,6,0],
    [7,8,2,6,4,1,0,0,5],
    [0,1,0,0,0,0,7,0,8]
]

hard_sudoku = [
    [8,0,0,0,0,0,0,0,0],
    [0,0,3,6,0,0,0,0,0],
    [0,7,0,0,9,0,2,0,0],
    [0,5,0,0,0,7,0,0,0],
    [0,0,0,0,4,5,7,0,0],
    [0,0,0,1,0,0,0,3,0],
    [0,0,1,0,0,0,0,6,8],
    [0,0,8,5,0,0,0,1,0],
    [0,9,0,0,0,0,4,0,0],
]

run_test("invalid", invalid_sudoku)
run_test("easy", easy_sudoku)
run_test("hard", hard_sudoku)

invalid: solved = False, valid = False, 0.000s 


easy: solved = True, valid = True, 0.000s 

5 2 8 | 4 6 7 | 3 1 9 
9 6 3 | 8 1 5 | 4 2 7 
1 7 4 | 2 9 3 | 5 8 6 
- - - - - - - - - - - 
2 3 1 | 9 7 6 | 8 5 4 
8 5 7 | 1 2 4 | 6 9 3 
4 9 6 | 3 5 8 | 1 7 2 
- - - - - - - - - - - 
3 4 5 | 7 8 9 | 2 6 1 
7 8 2 | 6 4 1 | 9 3 5 
6 1 9 | 5 3 2 | 7 4 8 

hard: solved = True, valid = True, 0.929s 

8 1 2 | 7 5 3 | 6 4 9 
9 4 3 | 6 8 2 | 1 7 5 
6 7 5 | 4 9 1 | 2 8 3 
- - - - - - - - - - - 
1 5 4 | 2 3 7 | 8 9 6 
3 6 9 | 8 4 5 | 7 2 1 
2 8 7 | 1 6 9 | 5 3 4 
- - - - - - - - - - - 
5 2 1 | 9 7 4 | 3 6 8 
4 3 8 | 5 2 6 | 9 1 7 
7 9 6 | 3 1 8 | 4 5 2 



In [85]:
N_BOX_ROWS = 2
N_BOX_COLS = 3
N_BOARD_ROWS = N_BOX_ROWS*N_BOX_COLS
N_BOARD_COLS = N_BOARD_ROWS

BOXES_PER_ROW = N_BOARD_ROWS//N_BOX_ROWS
BOXES_PER_COL = N_BOARD_COLS//N_BOX_COLS

junior_sudoku = [
    [5,0,0,0,0,0],
    [0,4,0,0,0,0],
    [0,0,3,0,0,0],
    [0,0,0,2,0,0],
    [0,0,0,0,1,0],
    [0,0,0,0,0,6],
]

run_test("junior", junior_sudoku)

junior: solved = True, valid = True, 0.017s 

5 3 1 | 4 6 2 
6 4 2 | 1 3 5 
- - - - - - - 
2 5 3 | 6 4 1 
1 6 4 | 2 5 3 
- - - - - - - 
3 2 6 | 5 1 4 
4 1 5 | 3 2 6 

